# 03. 법률 API 구조 기반 청킹

`02_preprocess_legal_api.ipynb` 결과를 읽어 법령·해석례·판례 구조에 맞게 청킹한다.
섹션 경계를 우선하고 길이 제한은 긴 본문에만 적용한다.


## 0. 경로와 청킹 설정

숫자는 글자 수 기준이며 `token_len`도 함께 기록해 분포를 확인한다.


In [ ]:
# 경로와 유형별 청킹 기준
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_DIR = ROOT / "data" / "03_processed" / "legal_api"
OUTPUT_DIR = ROOT / "data" / "04_chunks" / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_CONFIG = {
    "statute": {"max_chars": 1000, "overlap": 80},
    "interpretation": {"max_chars": 1200, "overlap": 120},
    "precedent": {"max_chars": 1500, "overlap": 150},
}
MAX_EMBEDDING_TOKENS = 8192

try:
    TOKENIZER = tiktoken.encoding_for_model("text-embedding-3-small")
except KeyError:
    TOKENIZER = tiktoken.get_encoding("cl100k_base")

print("입력:", INPUT_DIR)
print("출력:", OUTPUT_DIR)
print("설정:", CHUNK_CONFIG)


## 1. 전처리 결과 로드

`relevant`, `candidate` 판례만 전처리 파일에 들어 있으며 제외 판례는 별도 파일로 남는다.


In [ ]:
# 전처리 JSONL 로드
def read_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))
    return records


inputs = {
    "statute": read_jsonl(INPUT_DIR / "processed_eflaw.jsonl"),
    "interpretation": read_jsonl(INPUT_DIR / "processed_expc.jsonl"),
    "precedent": read_jsonl(INPUT_DIR / "processed_prec.jsonl"),
}
display(pd.DataFrame([
    {"type": source_type, "records": len(records)}
    for source_type, records in inputs.items()
]))


## 2. 공통 분할과 헤더 함수

각 청크에 출처 헤더를 반복해 metadata를 보지 않아도 문맥을 알 수 있게 한다.


In [ ]:
# 구조 경계를 우선하는 공통 함수
STATUTE_SEPARATORS = [
    "\n①", "\n②", "\n③", "\n④", "\n⑤", "\n⑥", "\n⑦", "\n⑧", "\n⑨", "\n⑩",
    "\n1.", "\n2.", "\n3.", "\n가.", "\n나.", "\n다.", "\n\n", "\n", ". ", " ", "",
]
PARAGRAPH_SEPARATORS = ["\n\n", "\n", ". ", "。", " ", ""]


def split_text(text: str, max_chars: int, overlap: int, separators: list[str]):
    if len(text) <= max_chars:
        return [text]
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars,
        chunk_overlap=overlap,
        length_function=len,
        separators=separators,
        keep_separator=True,
    )
    return splitter.split_text(text)


def split_table_rows(text: str, max_chars: int, overlap: int):
    """별표 행을 유지하되 제한보다 긴 단일 행만 재귀적으로 분할한다."""
    if len(text) <= max_chars:
        return [text]
    lines = [line for line in text.splitlines() if line.strip()]
    header = lines[:2]
    rows = lines[2:] if len(lines) > 2 else lines
    chunks, current = [], list(header)
    header_text = "\n".join(header).strip()
    available = max(1, max_chars - len(header_text) - 1)
    for row in rows:
        # 헤더를 반복하면서 남은 공간이 줄어도 행 자체가 제한 이하면 보존한다.
        # 제한을 넘는 단일 행만 헤더를 포함할 수 있는 길이로 재분할한다.
        if len(row) > max_chars:
            if current != header:
                chunks.append("\n".join(current).strip())
            row_overlap = min(overlap, max(0, available // 4))
            for part in split_text(
                row, available, row_overlap, PARAGRAPH_SEPARATORS
            ):
                chunks.append("\n".join([*header, part]).strip())
            current = list(header)
            continue

        candidate = "\n".join([*current, row])
        if current != header and len(candidate) > max_chars:
            chunks.append("\n".join(current).strip())
            current = [*header, row]
        else:
            current.append(row)
    if current != header or not chunks:
        chunks.append("\n".join(current).strip())
    return chunks


def build_header(metadata: dict) -> str:
    source_type = metadata["source_type"]
    if source_type == "statute":
        lines = [f"[법령] {metadata['doc_title']}"]
        if metadata.get("article"):
            title = f"({metadata['article_title']})" if metadata.get("article_title") else ""
            lines.append(f"[조문] {metadata['article']}{title}")
        else:
            lines.append(f"[구분] {metadata.get('section_title', metadata['section'])}")
        return "\n".join(lines)
    if source_type == "interpretation":
        return "\n".join(filter(None, [
            f"[법령해석례] {metadata['doc_title']}",
            f"[안건번호] {metadata.get('case_no', '')}",
            f"[구분] {metadata.get('section_title', metadata['section'])}",
        ]))
    return "\n".join(filter(None, [
        f"[판례] {metadata['doc_title']}",
        f"[법원] {metadata.get('court', '')}",
        f"[사건번호] {metadata.get('case_no', '')}",
        f"[구분] {metadata.get('section_title', metadata['section'])}",
    ]))


## 3. 유형별 청킹

짧은 조문·질의요지·회답·판시사항·판결요지는 통째로 유지한다.
이유와 판례내용 등 긴 본문만 항 또는 문단 경계로 나눈다.


In [ ]:
# 유형별 분할 규칙
def split_record(record: dict) -> list[str]:
    text = record["page_content"].strip()
    metadata = record["metadata"]
    source_type = metadata["source_type"]
    section = metadata["section"]
    config = CHUNK_CONFIG[source_type]

    if source_type == "statute":
        if section == "annex":
            return split_table_rows(
                text, config["max_chars"], config["overlap"]
            )
        return split_text(text, config["max_chars"], config["overlap"], STATUTE_SEPARATORS)

    if source_type == "interpretation":
        if section in {"question", "answer"}:
            return [text]
        return split_text(text, config["max_chars"], config["overlap"], PARAGRAPH_SEPARATORS)

    if section in {"holding", "summary"}:
        return [text]
    return split_text(text, config["max_chars"], config["overlap"], PARAGRAPH_SEPARATORS)


def build_chunks(records: list[dict]) -> list[dict]:
    chunks = []
    source_indexes = defaultdict(int)
    for record in records:
        metadata = record["metadata"]
        header = build_header(metadata)
        for section_index, body in enumerate(split_record(record)):
            content = f"{header}\n\n{body}".strip()
            source_id = metadata["source_id"]
            chunk_index = source_indexes[source_id]
            source_indexes[source_id] += 1
            chunk_metadata = {
                **metadata,
                "chunk_id": f"{metadata['record_id']}:{section_index}",
                "chunk_index": chunk_index,
                "section_chunk_index": section_index,
                "char_len": len(content),
                "token_len": len(TOKENIZER.encode(content)),
            }
            chunks.append({"page_content": content, "metadata": chunk_metadata})
    return chunks


chunk_sets = {
    source_type: build_chunks(records)
    for source_type, records in inputs.items()
}
display(pd.DataFrame([
    {"type": source_type, "input_records": len(inputs[source_type]), "chunks": len(chunks)}
    for source_type, chunks in chunk_sets.items()
]))


## 4. 청크 품질 검증

빈 청크, ID 중복, metadata 누락과 길이 분포를 확인한다. 통째로 유지한 요약 섹션은
설정 길이를 넘을 수 있으므로 별도 경고로 확인한다.


In [ ]:
# 청크 ID와 길이 검증
REQUIRED_METADATA = {
    "source_type", "source_id", "record_id", "doc_title", "section",
    "chunk_id", "chunk_index", "section_chunk_index", "char_len", "token_len",
}

quality_rows = []
for source_type, chunks in chunk_sets.items():
    chunk_ids = [item["metadata"]["chunk_id"] for item in chunks]
    empty = sum(not item["page_content"].strip() for item in chunks)
    missing = sum(
        bool(REQUIRED_METADATA - set(item["metadata"])) for item in chunks
    )
    duplicate = len(chunk_ids) - len(set(chunk_ids))
    lengths = [item["metadata"]["char_len"] for item in chunks]
    token_lengths = [item["metadata"]["token_len"] for item in chunks]
    quality_rows.append({
        "type": source_type,
        "chunks": len(chunks),
        "empty": empty,
        "missing_metadata": missing,
        "duplicate_chunk_id": duplicate,
        "char_p50": int(pd.Series(lengths).quantile(0.5)),
        "char_p95": int(pd.Series(lengths).quantile(0.95)),
        "char_max": max(lengths),
        "token_p95": int(pd.Series(token_lengths).quantile(0.95)),
        "token_max": max(token_lengths),
    })
    assert empty == 0
    assert missing == 0
    assert duplicate == 0
    assert max(token_lengths) <= MAX_EMBEDDING_TOKENS, (
        f"{source_type} 청크가 임베딩 토큰 한도를 초과했습니다: {max(token_lengths)}"
    )

quality = pd.DataFrame(quality_rows)
display(quality)

for source_type, chunks in chunk_sets.items():
    longest = max(chunks, key=lambda item: item["metadata"]["token_len"])
    print(
        f"[{source_type}] 최대 토큰={longest['metadata']['token_len']}, "
        f"제목={longest['metadata']['doc_title']}, 섹션={longest['metadata']['section']}"
    )
    print(longest["page_content"][:500], "\n")


## 5. 최종 JSONL 저장

기존 임베딩 파이프가 읽는 `page_content + metadata` 형식으로 유형별 저장한다.


In [ ]:
# 유형별 청크 JSONL 저장
OUTPUT_NAMES = {
    "statute": "kb_chunks_eflaw.jsonl",
    "interpretation": "kb_chunks_expc.jsonl",
    "precedent": "kb_chunks_prec.jsonl",
}


def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8", newline="\n") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


for source_type, chunks in chunk_sets.items():
    path = OUTPUT_DIR / OUTPUT_NAMES[source_type]
    write_jsonl(path, chunks)
    print(f"{source_type}: {len(chunks)}개 → {path}")
